# 9 · Chemistry: equilibrium constants

The 21 equilibria come from Herrmann et al. (2023). Any log K can be changed with a file (`logk_file`, one `name value` per line), for example values fitted to pH measurements, and the whole set can be read from a PHREEQC database instead (`equilibria_db`).

Activities are ideal (concentrations). At the cell's ionic strength (about 8 mol/L) the Davies-type models are outside their range; a Pitzer model is an improvement item.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model
import tempfile, os
from znmno2_model.eqchem import Equilibria

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

## The pH of the electrolyte against the ZnSO₄ complex constants

In [ ]:
base = Equilibria()
names = ("ZnSO4", "Zn(SO4)2-2", "HSO4-")
for nm in names:
    k0 = base.logk[base.names.index(nm)]
    phs = []
    for dk in (-0.5, 0.0, 0.5):
        eq = Equilibria({nm: k0 + dk})
        x = eq.solve(np.array([[0.0, 2.0, 0.05, 2.05]]))
        phs.append(-x[0, 0])
    print(f"{nm:12s} log K {k0:5.2f} -0.5 / 0 / +0.5:  pH {phs[0]:.3f} / {phs[1]:.3f} / {phs[2]:.3f}")

## Effect on a cycle

In [ ]:
def with_logk(overrides):
    f = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
    f.write("\n".join(f"{k} {v}" for k, v in overrides.items())); f.close()
    return f.name

k = Equilibria().logk[Equilibria().names.index("HSO4-")]
runs = {"literature": run(Params(steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False))}
for dk in (-0.5, 0.5):
    runs[f"HSO4- log K {k + dk:.2f}"] = run(Params(logk_file=with_logk({"HSO4-": k + dk}), steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False))
summary(runs)
compare(runs, "HSO4- formation constant");

## ZHS solubility

Herrmann et al. give log K_sp = 28.4; Bernard et al. (2025, GITT) found 30.5 ± 0.5 fitted the rest voltages better.

In [ ]:
runs = {f"logK_ZHS = {k}": run(Params(logK_ZHS=k, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for k in (28.4, 29.5, 30.5)}
summary(runs)
compare(runs, "ZHS solubility product", cols=("V", "pH_cath", "vf_ZHS"));

## A PHREEQC database

With a PHREEQC database file (for example `phreeqc.dat` from the USGS distribution), the equilibria of the species made of H⁺, Zn²⁺, Mn²⁺, SO₄²⁻ and water are read from it:

```python
r = run(Params(equilibria_db="/path/to/phreeqc.dat", steps="cc I=100 Vmin=1.0"))
```